[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/generic/generic.ipynb)

# Damping as Heating: a GENERIC System

This example computes how a mass on a spring, slowed by a damper, hands its energy to the damper as heat. The oscillation's kinetic and spring energy fall together while the damper's internal energy rises by just as much, so the total energy holds, and the damper's entropy grows until the mass comes to rest with all the energy in the damper, warmed. The GENERIC framework, the general equation for the non-equilibrium reversible–irreversible coupling, writes such a system as two motions: a reversible one driven by the energy and an irreversible one driven by the entropy.

A state is modelled as the mass's position and momentum and the damper's entropy, an antivector of the space spanned by `mv.q.dual()`, `mv.p.dual()` and `mv.s.dual()`. The energy's gradient, the spring's force, the mass's velocity and the damper's temperature, pairs with a change of state to the change of energy, and the entropy's gradient is `mv.s`. Two maps from gradients to changes of state drive the motion. The Poisson map, `mv.q.dual() * (Gradient & mv.p.dual()) - mv.p.dual() * (Gradient & mv.q.dual())`, is skew: it sends the energy's gradient to the swing of the oscillator and the entropy's gradient to nothing. The friction mobility is a positive dyad along the direction in which the momentum lost is the damper's heat gained: it sends the entropy's gradient to the damping and the energy's gradient to nothing. Those two null readings are why the energy holds and the entropy never falls.

In the matrix notation of the GENERIC literature, where $dE$ and $dS$ are the energy's and the entropy's gradients as columns, the equations of motion read $\dot x = L\,dE + M\,dS$, with $L$ antisymmetric, $M$ symmetric and positive, and the degeneracies $L\,dS = 0$ and $M\,dE = 0$; for this oscillator they are $m\ddot q = -kq - \gamma \dot q$ with $T\dot S = \gamma \dot q^2$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import jax
import numpy as np

from numga import Algebra
from numga.backend.jax import JaxContext, derivative
from examples.util.animation import show_animation
from examples.thermodynamics.generic import render

ga = Algebra("q0p0s0")
mv = JaxContext(ga, np.float64).multivector
Scalar = ga.gatype.scalar()                    # Scalar+
State = ga.gatype.antivector()                 # Antivector-
Gradient = ga.gatype.vector()                  # Vector+
Map = ga.gatype((State, Gradient))             # State- <- Gradient+

# The mass, the spring's stiffness and the damper's friction; the damper's heat capacity and starting temperature;
# the mass's starting displacement.
mass, stiffness, friction = 1.0, 4.0, 0.35
thermal_capacity, reference_temperature = 2.0, 1.0
displacement = 2.0

## The energy and the two maps

The energy is the kinetic energy, the spring's, and the damper's internal energy, its heat capacity times its temperature, which grows exponentially with its entropy. The energy's gradient holds its slopes along the three directions. The Poisson map turns it into the reversible motion; the friction mobility, a dyad along the direction in which the momentum the mass loses becomes the damper's heat, a direction the energy's gradient reads as nothing, turns the entropy's gradient into the irreversible motion. The mass starts displaced and at rest.

At fixed total energy the damper's entropy is a function of the position and momentum alone, `surface_entropy`: the damper holds whatever energy the motion leaves over. Its level lines on the phase portrait are the ellipses of equal mechanical energy, marked with their entropy. The reversible motion runs along them, since the Poisson map sends the entropy's gradient to nothing; the irreversible motion crosses them inward, up the entropy. The second chart is the entropy against time, the third the entropy against the mechanical energy left, along the fixed total energy.

In [ ]:
poisson_map = (mv.q.dual() * (Gradient & mv.p.dual())
               - mv.p.dual() * (Gradient & mv.q.dual()))                       # [] State- <- Gradient+
entropy_gradient = mv.s                                                       # [] Gradient+

def temperature(state: State) -> Scalar:
    return reference_temperature * ((mv.s & state) / thermal_capacity).exp()

def energies(state: State) -> tuple[Scalar, Scalar, Scalar]:
    """The kinetic energy, the spring's and the damper's internal energy."""
    return ((mv.p & state).squared() / (2 * mass), stiffness * (mv.q & state).squared() / 2,
            thermal_capacity * temperature(state))

def energy_gradient(state: State) -> Gradient:
    """The energy's slopes: the spring's force, the velocity, the temperature."""
    return mv.q * (stiffness * (mv.q & state)) + mv.p * ((mv.p & state) / mass) + mv.s * temperature(state)

def friction_mobility(state: State) -> Map:
    """A positive dyad along the direction in which the momentum lost is the damper's heat gained."""
    damper_temperature = temperature(state)
    direction = -mv.p.dual() + mv.s.dual() * ((mv.p & state) / mass / damper_temperature)
    return direction * (Gradient & direction) * (friction * damper_temperature)

def rates(state: State) -> tuple[State, State]:
    """The reversible motion and the irreversible."""
    return poisson_map(energy_gradient(state)), friction_mobility(state)(entropy_gradient)

start = mv.q.dual() * displacement + mv.p.dual() * 0.0 + mv.s.dual() * 0.0     # [] State-
total_energy = sum(energies(start))                                           # [] Scalar+

def surface_entropy(state: State) -> Scalar:
    """The damper's entropy at the start's total energy, as a function of the position and momentum: the damper holds
    all the energy they leave over."""
    kinetic, potential, _ = energies(state)
    internal = total_energy - kinetic - potential
    return thermal_capacity * (internal / (thermal_capacity * reference_temperature)).log()

render.draw(energies, surface_entropy, rates, start);

## The release

The two motions together are integrated by the classical fourth-order Runge–Kutta method, compiled by JAX, a frame every two steps.

This cell takes about ten seconds, nearly all of it drawing the animation.

In [ ]:
@jax.jit
def advance(state: State, dt: float) -> State:
    """A step of the combined motion by the classical fourth-order Runge–Kutta method."""
    def rate(current: State) -> State:
        reversible, irreversible = rates(current)
        return reversible + irreversible
    k1 = rate(state)
    k2 = rate(state + k1 * (dt / 2))
    k3 = rate(state + k2 * (dt / 2))
    k4 = rate(state + k3 * dt)
    return state + (k1 + 2 * k2 + 2 * k3 + k4) * (dt / 6)

dt, steps_per_frame, n_frames, frame_duration_ms = 0.04, 2, 150, 40

def release() -> Iterator[State]:
    """The state at each frame."""
    state = start
    for _ in range(n_frames):
        yield state
        for _ in range(steps_per_frame):
            state = advance(state, dt)

show_animation(render.damping(energies, surface_entropy, rates, release(), steps_per_frame * dt),
               "generic_thermodynamics", duration_ms=frame_duration_ms)

The state spirals inward, up across the levels of the entropy: the blue arrow runs along its level and the red one crosses it inward, largest where the mass moves fastest. The entropy rises in steps, fast as the mass passes the middle and flat at its turning points, and never falls; on the third chart the state slides up the curve of fixed total energy to its top, where the damper holds all of it. The total energy stays at 10, of which 2 starts in the damper; at rest all 10 is in the damper, at the temperature 5 for its heat capacity of 2. The damper's colour runs from icy blue to red heat by the warming it has still to do, half as it is and half on its logarithm, so its last warming stays visible.

In [ ]:
end = list(release())[-1]                                                     # [] State
final_energies = energies(end)

In [ ]:
print("final kinetic, spring and internal energy:", np.array([energy.to_array() for energy in final_energies]))
print("final temperature:", temperature(end).to_array())

In [ ]:
# checks
total = lambda state: sum(energies(state))
totals = [total(state).to_array() for state in release()]
np.testing.assert_allclose(totals, 10, atol=1e-4)
assert np.all(np.diff([(mv.s & state).to_array() for state in release()]) >= -1e-10)
# The two degeneracies: the Poisson map ignores the entropy, the friction mobility the energy.
moving = mv.q.dual() * 0.7 + mv.p.dual() * -1.3 + mv.s.dual() * 0.4
np.testing.assert_allclose(poisson_map(entropy_gradient).kernel, 0, atol=1e-10)
np.testing.assert_allclose(friction_mobility(moving)(energy_gradient(moving)).kernel, 0, atol=1e-10)
# The energy's gradient, as a reading of changes of state, is the energy's derivative.
np.testing.assert_allclose(((energy_gradient(moving) & State) - derivative(total)(moving)).kernel,
                           0, atol=1e-10)